# From Analysis to Dashboard
### ECON 148 — Data Science for Economics

In Notebook 1 you built a backtesting pipeline from scratch. The problem: to change the ticker or threshold, you had to edit code and rerun cells. That's fine for you. It's not fine for anyone else.

This notebook walks through two levels of turning that analysis into something interactive — using only `ipywidgets`, which works reliably on JupyterHub without any extra server setup:

| Level | Tool | What you write | What the user sees |
|---|---|---|---|
| 1 | `@interact` | Regular notebook + decorator | Sliders and dropdowns, auto-reactive |
| 2 | `HBox` / `VBox` + `Output` | Manual widget layout | A proper sidebar + main panel dashboard |

Both run entirely inside the notebook kernel — no Voilà, no Panel, no extra server.

---
> **Prerequisite:** Run Notebook 1 first, or run the setup cells below to get `prices`, `train`, and `test` in memory.

## 0. Setup & Data

We'll reinstall the same dependencies and re-fetch the data so this notebook is self-contained.

In [ ]:
import sys
!{sys.executable} -m pip install yfinance statsmodels xgboost --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import timedelta

import yfinance as yf
import xgboost as xgb
import ipywidgets as widgets
from ipywidgets import interact, Output, VBox, HBox, Layout
from IPython.display import display, clear_output

plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor':   '#ffffff',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        11,
})

print('Ready.')

In [ ]:
# ── Configuration ────────────────────────────────
TICKER     = 'AAPL'
START_DATE = '2021-01-01'
END_DATE   = '2025-12-31'
TEST_FRAC  = 0.20
# ─────────────────────────────────────────────────

raw = yf.download(TICKER, start=START_DATE, end=END_DATE,
                  auto_adjust=True, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

prices = raw[['Open','High','Low','Close','Volume']].dropna()
split  = int(len(prices) * (1 - TEST_FRAC))
train  = prices.iloc[:split]
test   = prices.iloc[split:]

print(f'{TICKER}: {len(prices)} days, split at {test.index[0].date()}')

---
## 1. Clean Functions First

Before we can make anything interactive, we need to **refactor the analysis into functions**. In Notebook 1, everything was written procedurally — one cell per step. Here we wrap each step so it takes inputs and returns outputs.

This is the prerequisite for any dashboard framework. Widgets call functions. Functions need to be callable.

In [ ]:
def fetch_prices(ticker: str, start: str, end: str,
                 test_frac: float = 0.2) -> tuple:
    """Download OHLCV data and return (prices, train, test)."""
    raw = yf.download(ticker, start=start, end=end,
                      auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    prices = raw[['Open','High','Low','Close','Volume']].dropna()
    split  = int(len(prices) * (1 - test_frac))
    return prices, prices.iloc[:split], prices.iloc[split:]


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineer lag and rolling features from OHLCV data."""
    X      = pd.DataFrame(index=df.index)
    close  = df['Close']
    volume = df['Volume']
    ret    = close.pct_change()

    for lag in [1, 2, 3, 5, 10, 21]:
        X[f'ret_lag{lag}'] = ret.shift(lag)
    for window in [5, 10, 21]:
        X[f'roll_mean_{window}'] = close.rolling(window).mean() / close - 1
        X[f'roll_std_{window}']  = ret.rolling(window).std()
    X['mom_1m']     = close / close.shift(21) - 1
    X['mom_3m']     = close / close.shift(63) - 1
    X['vol_ratio']  = volume / volume.rolling(21).mean()
    X['day_of_week']= pd.to_datetime(df.index).dayofweek
    X['target']     = ret.shift(-1)
    return X.dropna()


def fit_xgb_forecast(train_: pd.DataFrame, test_: pd.DataFrame):
    """
    Fit XGBoost on engineered features from train_, predict on test_.
    Returns (predicted_returns, aligned_test_prices).
    """
    feat_all   = make_features(pd.concat([train_, test_]))
    feat_cols  = [c for c in feat_all.columns if c != 'target']
    train_feat = feat_all[feat_all.index < test_.index[0]]
    test_feat  = feat_all[feat_all.index >= test_.index[0]]

    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    )
    model.fit(train_feat[feat_cols], train_feat['target'])

    preds          = pd.Series(model.predict(test_feat[feat_cols]), index=test_feat.index)
    prices_aligned = test_.loc[test_feat.index, 'Close']
    return preds, prices_aligned, model, feat_cols


def backtest(predicted_returns: pd.Series,
             actual_prices:    pd.Series,
             threshold:        float = 0.0) -> dict:
    """
    Convert predicted returns -> long/cash signal -> equity curve + metrics.
    Signal is shifted by 1 day to prevent look-ahead bias.
    """
    actual_returns = actual_prices.pct_change().fillna(0)
    signal         = (predicted_returns > threshold).astype(int)
    signal_lagged  = signal.shift(1).fillna(0)
    strat_returns  = signal_lagged * actual_returns

    equity_strat = (1 + strat_returns).cumprod()
    equity_bnh   = (1 + actual_returns).cumprod()

    n       = len(strat_returns)
    ann_ret = (equity_strat.iloc[-1] ** (252 / n) - 1) * 100
    ann_vol = strat_returns.std() * np.sqrt(252) * 100
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0
    max_dd  = ((equity_strat - equity_strat.cummax()) / equity_strat.cummax()).min() * 100

    return dict(
        equity   = equity_strat,
        bnh      = equity_bnh,
        ann_ret  = ann_ret,
        ann_vol  = ann_vol,
        sharpe   = sharpe,
        max_dd   = max_dd,
        pct_long = signal_lagged.mean() * 100,
        bnh_ret  = (equity_bnh.iloc[-1] ** (252 / n) - 1) * 100,
    )


def plot_equity(result: dict, title: str, ax=None):
    """Plot strategy equity curve vs buy-and-hold."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(result['equity'].index, result['equity'],
            lw=2, color='#16a34a', label='XGBoost Strategy')
    ax.plot(result['bnh'].index, result['bnh'],
            lw=2, ls='--', color='#6b7280', label='Buy & Hold')
    ax.axhline(1, color='#d1d5db', lw=1)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Portfolio Value')
    ax.legend(fontsize=10)
    return ax


print('Functions defined.')


Notice how each function has a clear **input → output** contract. That's what makes the next step possible.

---
## 2. Level 1 — `@interact`: Sliders Inside the Notebook

`ipywidgets.interact` wraps a function and auto-generates UI controls from its arguments. Pass an integer range and you get a slider. Pass a list and you get a dropdown. That's it.

**This is reactivity in a notebook.** The function reruns every time you move a slider — no cell rerun needed.

### 2a. Simple example: just the price chart

In [ ]:
@interact(
    ticker  = widgets.Dropdown(options=['AAPL','NVDA','TSLA','SPY','GLD','BTC-USD'], value='AAPL'),
    years   = widgets.IntSlider(min=1, max=5, value=3, description='Years back'),
)
def show_price(ticker, years):
    from datetime import date
    end   = date.today().isoformat()
    start = (date.today() - timedelta(days=years*365)).isoformat()

    prices_, _, _ = fetch_prices(ticker, start, end)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(prices_.index, prices_['Close'], lw=1.5, color='#2563eb')
    ax.set_title(f'{ticker} — Adjusted Close', fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.show()

### 2b. XGBoost backtest with `@interact`

Now the full XGBoost pipeline — feature engineering, fitting, backtesting — wired to sliders.
XGBoost fits in seconds, so the widget responds immediately when you change parameters.

Try adjusting `max_depth` and `n_estimators` and watch what happens to the equity curve.
Is a deeper tree always better?

In [ ]:
def xgb_backtest_plot(ticker='AAPL', threshold_pct=0.0):
    """
    Fetch data for ticker, fit XGBoost with default hyperparameters,
    backtest and plot results.
    """
    prices_, train_, test_ = fetch_prices(ticker, '2021-01-01', '2025-12-31')

    feat_all   = make_features(pd.concat([train_, test_]))
    feat_cols  = [c for c in feat_all.columns if c != 'target']
    train_feat = feat_all[feat_all.index < test_.index[0]]
    test_feat  = feat_all[feat_all.index >= test_.index[0]]

    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    )
    model.fit(train_feat[feat_cols], train_feat['target'])
    preds          = pd.Series(model.predict(test_feat[feat_cols]), index=test_feat.index)
    prices_aligned = test_.loc[test_feat.index, 'Close']
    result         = backtest(preds, prices_aligned, threshold=threshold_pct / 100)

    # Feature importance
    importance = pd.Series(
        model.feature_importances_, index=feat_cols
    ).sort_values(ascending=True).tail(10)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    plot_equity(result, f'{ticker} — XGBoost Strategy', ax=axes[0])

    importance.plot(kind='barh', ax=axes[1], color='#86efac')
    axes[1].set_title('Feature Importance (top 10)', fontweight='bold')
    axes[1].set_xlabel('Importance')

    axes[2].axis('off')
    table_data = [
        ['Annualized Return', f"{result['ann_ret']:+.1f}%"],
        ['Annualized Vol',    f"{result['ann_vol']:.1f}%"],
        ['Sharpe Ratio',      f"{result['sharpe']:.2f}"],
        ['Max Drawdown',      f"{result['max_dd']:.1f}%"],
        ['% Days Long',       f"{result['pct_long']:.0f}%"],
        ['Buy & Hold',        f"{result['bnh_ret']:+.1f}%"],
    ]
    tbl = axes[2].table(cellText=table_data, colLabels=['Metric', 'Value'],
                        loc='center', cellLoc='left')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.2, 1.8)
    axes[2].set_title('Performance Metrics', fontweight='bold', pad=20)

    plt.tight_layout()
    plt.show()


interact(
    xgb_backtest_plot,
    ticker        = widgets.Dropdown(
                        options=['AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA',
                                 'SPY','GLD','BTC-USD'],
                        value='AAPL',
                        description='Ticker',
                    ),
    threshold_pct = widgets.FloatSlider(
                        min=0.0, max=1.0, step=0.1, value=0.0,
                        description='Threshold %',
                        readout_format='.1f',
                    ),
);


### Discussion — `@interact`

- What changed in the code between the raw notebook and the `@interact` version? Almost nothing — we just moved the analysis into a function and added a decorator.
- Change the ticker to TSLA or BTC-USD. Does XGBoost beat buy-and-hold? Does that surprise you?
- The `threshold_pct` slider means: only go long if the predicted return exceeds that threshold. Try 0.1% and 0.5%. Does being more selective help?
- `@interact` reruns the entire function on every widget change. What are the implications for slow computations?

---
## 3. Level 2 — Manual Widget Layout: Sidebar + Main Panel

`@interact` is convenient but gives you no control over layout — everything stacks vertically and reruns on every change. Sometimes you want:

- A **sidebar** with controls on the left and results on the right
- A **button** to trigger the analysis (instead of running on every slider move)
- Multiple **output areas** updating independently

You can build all of that with three ipywidgets primitives:

- `Output()` — a container that captures anything you `display()` or `print()` into it
- `VBox([...])` — stack widgets vertically
- `HBox([...])` — arrange widgets horizontally (this is your sidebar + main split)

The pattern is: define your widgets, define an `Output` pane for each region, wire a button's `on_click` to a function that clears and redraws the outputs, then arrange everything with `HBox`/`VBox`.

In [ ]:
# ── Quick demo: HBox and VBox ─────────────────────────────────────────
# Before building the full dashboard, here's the layout primitive in isolation.

left  = widgets.Textarea(value='I am the sidebar', layout=Layout(width='200px', height='80px'))
right = widgets.Textarea(value='I am the main panel', layout=Layout(width='400px', height='80px'))

display(HBox([left, right]))
print('Drag the boundary — this is your sidebar/main split.')

---
## 4. The Full Dashboard

Now we put it together: a ticker dropdown and threshold slider in a sidebar, a button to trigger the analysis, and two output panes (equity chart + metrics table) in the main panel.

The key pattern is `on_click` — the analysis only runs when you press the button, not on every widget change. This matters when computation takes more than a second.

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────
ticker_dd  = widgets.Dropdown(
    options=['AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA','SPY','GLD','BTC-USD'],
    value='AAPL',
    description='Ticker:',
    layout=Layout(width='220px'),
)
thresh_sl  = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.1, value=0.0,
    description='Threshold %:',
    readout_format='.1f',
    style={'description_width': 'initial'},
    layout=Layout(width='280px'),
)
run_btn    = widgets.Button(
    description='Run Backtest',
    button_style='primary',
    layout=Layout(width='160px', margin='12px 0 0 0'),
)
status_lbl = widgets.Label(value='Press Run to start.')

# Output panes
chart_out   = Output(layout=Layout(width='100%'))
metrics_out = Output(layout=Layout(width='100%'))


# ── Click handler ─────────────────────────────────────────────────────────
def on_run(b):
    ticker = ticker_dd.value
    status_lbl.value = f'Fetching {ticker}...'

    try:
        prices_, train_, test_ = fetch_prices(
            ticker, '2021-01-01', '2025-12-31', test_frac=0.2
        )
        status_lbl.value = 'Fitting XGBoost...'

        feat_all   = make_features(pd.concat([train_, test_]))
        feat_cols  = [c for c in feat_all.columns if c != 'target']
        train_feat = feat_all[feat_all.index < test_.index[0]]
        test_feat  = feat_all[feat_all.index >= test_.index[0]]

        model = xgb.XGBRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, verbosity=0,
        )
        model.fit(train_feat[feat_cols], train_feat['target'])
        preds          = pd.Series(model.predict(test_feat[feat_cols]), index=test_feat.index)
        prices_aligned = test_.loc[test_feat.index, 'Close']
        result         = backtest(preds, prices_aligned,
                                  threshold=thresh_sl.value / 100)

        # ── Chart output ──────────────────────────────────────────────────
        importance = pd.Series(
            model.feature_importances_, index=feat_cols
        ).sort_values(ascending=True).tail(10)

        with chart_out:
            clear_output(wait=True)
            fig, axes = plt.subplots(1, 2, figsize=(13, 4))
            plot_equity(result, f'{ticker} — XGBoost vs Buy & Hold', ax=axes[0])
            importance.plot(kind='barh', ax=axes[1], color='#86efac')
            axes[1].set_title('Feature Importance (top 10)', fontweight='bold')
            plt.tight_layout()
            plt.show()

        # ── Metrics output ────────────────────────────────────────────────
        with metrics_out:
            clear_output(wait=True)
            df_m = pd.DataFrame([
                {'Metric': 'Annualized Return', 'Value': f"{result['ann_ret']:+.1f}%"},
                {'Metric': 'Annualized Vol',    'Value': f"{result['ann_vol']:.1f}%"},
                {'Metric': 'Sharpe Ratio',      'Value': f"{result['sharpe']:.2f}"},
                {'Metric': 'Max Drawdown',      'Value': f"{result['max_dd']:.1f}%"},
                {'Metric': '% Days Long',       'Value': f"{result['pct_long']:.0f}%"},
                {'Metric': 'Buy & Hold',        'Value': f"{result['bnh_ret']:+.1f}%"},
            ]).set_index('Metric')
            display(df_m)

        status_lbl.value = f'\u2713 Done \u2014 {ticker} \u00b7 {len(test_feat)} test days'

    except Exception as e:
        import traceback
        status_lbl.value = f'Error: {e}'
        with chart_out:
            clear_output(wait=True)
            print(traceback.format_exc())


run_btn.on_click(on_run)

# ── Layout ────────────────────────────────────────────────────────────────
sidebar = VBox(
    [widgets.HTML('<b>Controls</b>'), ticker_dd, thresh_sl, run_btn, status_lbl],
    layout=Layout(width='300px', padding='10px', margin='0 20px 0 0',
                  border='1px solid #e5e7eb'),
)
main = VBox(
    [widgets.HTML('<b>Results</b>'), chart_out,
     widgets.HTML('<b>Metrics</b>'), metrics_out],
    layout=Layout(flex='1'),
)

display(HBox([sidebar, main], layout=Layout(width='100%', align_items='flex-start')))

### Discussion — Manual Widget Layout

- Compare the `on_click` pattern to `@interact`. What's the key difference in when the function runs?
- `clear_output(wait=True)` prevents flickering by waiting until new content is ready before clearing. What would happen without it?
- We have two separate `Output` panes — `chart_out` and `metrics_out`. Why is that better than one shared output?
- This dashboard runs entirely inside the notebook kernel. What does that mean for sharing it with someone who doesn't have Python installed?

In [ ]:
# Nothing to run here — the dashboard above is self-contained.
# To share it: File > Download as > HTML (with outputs)
# or run the notebook on a shared JupyterHub and send the link.

---
## 5. The Full Stack — Putting It Together

Here's the progression from raw analysis to interactive tool:

```
yfinance + XGBoost     →  raw analysis in a notebook
@interact              →  sliders, auto-reactive, zero layout control
HBox / VBox / Output   →  button-triggered, sidebar layout, multiple output panes
Panel / Voilà          →  hides code, serves as URL (needs server config)
Shiny for Python       →  full UI control, production deployment
```

For most data science work in an academic setting, **`HBox`/`VBox` + `Output`** is the right stopping point. It works everywhere Jupyter works, requires no extra dependencies, and teaches the same reactive programming concepts as the heavier frameworks.

The heavier frameworks (Panel, Shiny) are worth knowing exist — but reach for them when you need to deploy to users who don't have Jupyter, not as a first instinct.

---
## 5. The Full Stack — Putting It Together

Here's the mental map of everything we've covered across both notebooks:

```
yfinance          →  raw data
ARIMA / XGBoost   →  predicted returns
backtest()        →  equity curve + metrics
@interact         →  sliders in the notebook
Voilà             →  hide the code, serve the widgets
Panel             →  real layout, button-triggered, servable
Shiny for Python  →  production app, full UI control
```

Each layer builds on the one below. You never had to learn a new analysis framework — just a new presentation layer.

### When to use what

**`@interact`** — you're exploring yourself, or demoing to someone watching over your shoulder.

**Voilà** — you want to share with someone non-technical on JupyterHub without rewriting anything.

**Panel** — you want a multi-panel layout, tabs, or to serve to multiple users simultaneously.

**Shiny** — you want full UI control (custom CSS, complex state, production deployment).

### The honest tradeoff

Voilà and Panel are fast to build but slow to customize. Shiny is slower to build but you control every pixel. For most data science use cases in an academic or research setting, Panel is exactly the right stopping point.

---
## Extension: Add a Comparison Tab

Panel supports tabs natively. Try extending the dashboard to show ARIMA vs XGBoost side by side.

In [ ]:
# Starter code — fill in the XGBoost tab
# Hint: use make_features() and xgb.XGBRegressor from Notebook 1

arima_tab   = pn.pane.Markdown('*ARIMA dashboard goes here — move the Panel layout from Section 4*')
xgboost_tab = pn.pane.Markdown('*Your turn: build the XGBoost version*')

tabs = pn.Tabs(
    ('ARIMA',   arima_tab),
    ('XGBoost', xgboost_tab),
)

tabs